# Market sentiment vs share-price performance

This is the market-sentiment half of the financial workstream. It answers the brief's question: does media coverage line up with how a company's shares perform?

It builds on the shared dataset from `financial_data.ipynb`. For each listed company it pulls, from GDELT (free, no key):
- **news volume**: how many articles mention the company,
- **news tone**: GDELT's own sentiment score, where positive means good-news coverage and negative means bad-news coverage,

then checks those against the one-year share-price return and volatility we already have.

**Honest caveats, worth stating in the write-up:**
- This is a small sample of listed firms (around 26). The correlations are indicative, not proof.
- Matching a company to its news by name is imperfect. Common names can pull in unrelated articles.
- This only covers listed firms. Private SMEs, the real target, have neither a share price nor much news, so this method does not reach them.

**How this fits with my teammate:** I work in this notebook (sentiment). My teammate works in a separate notebook on the financial ratios and a first model. We both load the same shared dataset, so we never edit the same file.

## Step 1: Load the shared dataset

In [ ]:
import pandas as pd

# Load the shared dataset built by financial_data.ipynb.
# Run that notebook first if this file is missing.
market = pd.read_csv("../data/processed/listed_companies.csv")
print(f"Loaded {len(market)} companies")
market[["ticker", "name", "sector", "return_1y", "volatility_1y"]].head()

## Step 2: News helpers (GDELT)

These small functions clean a company name into a search phrase, then ask GDELT for the article count and the average tone over the last year. They are rate limited and cached, so they are polite to the free service and fast to re-run.

In [ ]:
import json, re, time
from pathlib import Path
import requests

GDELT = "https://api.gdeltproject.org/api/v2/doc/doc"

# Words to drop from a company name so the news search is cleaner.
_DROP = {"limited", "ltd", "plc", "plc.", "ltd.", "llp", "group", "holdings",
         "the", "co", "uk", "ord", "company", "shs", "cls"}

_cache = Path("../data/raw/gdelt_cache")
_cache.mkdir(parents=True, exist_ok=True)
_last = [0.0]   # last call time, for the rate limit


def clean_name(name):
    """Turn a messy listed name into a short search phrase, e.g.
    'MARKS AND SPENCER GROUP PLC ORD' -> 'marks and spencer'."""
    n = re.sub(r"[^a-z0-9 ]+", " ", str(name).lower())
    toks = [w for w in n.split()
            if w not in _DROP and not any(ch.isdigit() for ch in w) and len(w) > 1]
    return " ".join(toks).strip()


def _throttle():
    # GDELT asks for no more than one request every 5 seconds.
    wait = 5.5 - (time.monotonic() - _last[0])
    if wait > 0:
        time.sleep(wait)
    _last[0] = time.monotonic()


def _gdelt(params):
    key = re.sub(r"[^A-Za-z0-9]+", "_", params["query"] + params["mode"])[:100]
    f = _cache / (key + ".json")
    if f.exists():
        return json.loads(f.read_text(encoding="utf-8"))
    s = requests.Session()
    s.headers.update({"User-Agent": "lloyds-student/1.0"})
    for _ in range(4):
        _throttle()
        r = s.get(GDELT, params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(6); continue
        if r.status_code == 200 and r.text.strip().startswith("{"):
            d = r.json(); f.write_text(json.dumps(d), encoding="utf-8"); return d
        if r.status_code == 200:
            f.write_text("{}", encoding="utf-8"); return {}
        time.sleep(6)
    return {}


def company_news(name, timespan="12months"):
    """Return news volume (article count) and average tone for a company.
    Tone is GDELT's own sentiment score: positive is good news, negative is bad."""
    cleaned = clean_name(name)
    if len(cleaned) < 3:
        return {"news_volume": 0, "news_tone": None}
    q = f'"{cleaned}"'
    art = _gdelt({"query": q, "mode": "artlist", "format": "json",
                  "maxrecords": "250", "timespan": timespan})
    volume = len((art or {}).get("articles", []))
    tone = _gdelt({"query": q, "mode": "timelinetone", "format": "json",
                   "timespan": timespan})
    pts = (tone.get("timeline") or [{}])[0].get("data", []) if tone else []
    vals = [p["value"] for p in pts if "value" in p]
    avg_tone = round(sum(vals) / len(vals), 3) if vals else None
    return {"news_volume": volume, "news_tone": avg_tone}

print("News helpers ready.")

## Step 3: Pull news volume and tone for each company

This is the slow cell (about 5 minutes the first time, instant after that thanks to the cache).

In [ ]:
# Pull news volume and tone for each company.
# This makes two GDELT calls per company and is rate limited to one call every
# 5.5 seconds, so for ~26 companies it takes about 5 minutes. Results are cached,
# so running it again is instant.
records = []
for row in market.itertuples(index=False):
    n = company_news(row.name)
    records.append({"ticker": row.ticker, **n})
    print(f"  {row.ticker:8s} volume={n['news_volume']:>4}  tone={n['news_tone']}")

news = pd.DataFrame(records)
market = market.merge(news, on="ticker", how="left")
market.to_csv("../data/processed/listed_companies_with_news.csv", index=False)
print(f"\nSaved companies with news to ../data/processed/listed_companies_with_news.csv")

## Step 4: Correlate coverage and tone with share-price performance

A positive correlation between tone and return would mean companies with more positive coverage tended to have stronger share prices over the year.

In [ ]:
# Do news coverage and tone line up with share-price performance?
cols = ["news_volume", "news_tone", "return_1y", "volatility_1y"]
feat = market[cols].dropna()
print(f"Companies with full data: {len(feat)}")
print()
print("Correlation matrix:")
print(feat.corr().round(2).to_string())
print()
print("Headline correlations:")
print(f"  news tone   vs 1y return    : {feat['news_tone'].corr(feat['return_1y']):.2f}")
print(f"  news volume vs 1y return    : {feat['news_volume'].corr(feat['return_1y']):.2f}")
print(f"  news tone   vs volatility   : {feat['news_tone'].corr(feat['volatility_1y']):.2f}")

## Step 5: A picture

Each point is a company. If positive news tone went with stronger returns, the points would slope up from bottom-left to top-right.

In [ ]:
import matplotlib.pyplot as plt

plot_df = market.dropna(subset=["news_tone", "return_1y"])
plt.figure(figsize=(7, 5))
plt.scatter(plot_df["news_tone"], plot_df["return_1y"])
for _, r in plot_df.iterrows():
    plt.annotate(r["ticker"].replace(".L", ""), (r["news_tone"], r["return_1y"]), fontsize=8)
plt.axhline(0, color="grey", lw=0.7)
plt.axvline(0, color="grey", lw=0.7)
plt.xlabel("Average news tone (positive = good news)")
plt.ylabel("1 year share-price return")
plt.title("News tone vs share-price return (UK-listed sample)")
plt.tight_layout()
plt.show()

## How to read the result, and what comes next

- If tone and return are positively correlated, that supports the brief's idea that media coverage carries a signal about performance. If the correlation is weak or noisy, that is also a real finding, especially given the small sample.
- This is a snapshot correlation. A stronger test is the lead-lag: does a change in tone come *before* a price move? That mirrors the lead-lag test used in the attrition work and is the natural next step.
- A richer sentiment score than GDELT's tone is FinBERT, a finance-tuned language model. That is an upgrade for later; GDELT tone keeps this first version simple and keyless.
- To make the correlation more reliable, widen the sample to more listed firms in `financial_data.ipynb`.